In [1]:
from pathlib import Path
import os

# zawsze najpierw lokalny katalog roboczy
# os.chdir("/content")

# mount point
DRIVE_MOUNT_PATH = Path("/content/drive")

WORKING_DIR = DRIVE_MOUNT_PATH / "MyDrive" / "Colab_Notebooks" / "GEQIE" 

# dataset directory
DATASETS_PATH = WORKING_DIR / "datasets" / "mnist"

# GEQIE directory
GEQIE_PATH = WORKING_DIR / "code" / "geqie"

# QCNN integration code directory
QNN_INTEGRATION_PATH = WORKING_DIR / "code" / "QCNN_integration"

try: 
	from google.colab import drive
	print("Environment is Google Colab.")
	print(f"--- GEQIE path set to '{GEQIE_PATH}'")
	print(f"--- MNIST path set to '{DATASETS_PATH}'")

	print(f"--- Mounting Google Drive at '{DRIVE_MOUNT_PATH}'...")
	drive.mount(str(DRIVE_MOUNT_PATH), force_remount=False)

	# po mountowaniu znów ustaw bezpieczny cwd
	os.chdir("/content")


	print("Installing dependencies...")
	%cd $GEQIE_PATH 
	!git pull # pull latest changes from the repo
	!pip install .
	!git checkout QNN # QNN branch
	%pip install -r {GEQIE_PATH}/requirements/requirements.txt # main geqie dependencies
	%pip install -r {GEQIE_PATH}/experiments/QNN_integration/requirements.txt # QNN integration dependencies

	print("Dependencies installed successfully. (restart may be required -> 'Restart runtime and run all')")
except ModuleNotFoundError as e:
	print("Environment is not Google Colab... skipping.")
except Exception as e:
	print("Error encountered: " + str(e))

mnist_train_dataset = "train-00000-of-00001.parquet"
mnist_test_dataset = "test-00000-of-00001.parquet"

path_to_train_mnist_dataset = DATASETS_PATH / mnist_train_dataset
path_to_test_mnist_dataset = DATASETS_PATH / mnist_test_dataset

# Checkpoint is persisted on Google Drive and updated only when validation loss improves
checkpoint_dir = WORKING_DIR / "checkpoints"
precomputed_circuits_dir = WORKING_DIR / "circuits"

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(precomputed_circuits_dir, exist_ok=True)

checkpoint_path = checkpoint_dir / "qnn_best_checkpoint.pt"


Environment is Google Colab.
--- GEQIE path set to '/content/drive/MyDrive/Colab_Notebooks/GEQIE/code/geqie'
--- MNIST path set to '/content/drive/MyDrive/Colab_Notebooks/GEQIE/datasets/mnist'
--- Mounting Google Drive at '/content/drive'...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Installing dependencies...
/content/drive/MyDrive/Colab_Notebooks/GEQIE/code/geqie
Already up to date.
Processing /content/drive/MyDrive/Colab_Notebooks/GEQIE/code/geqie
  Preparing metadata (setup.py) ... done
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
Using cached dill-0.4.0-py3-none-any.whl (119 kB)
  Created wheel for geqie: filename=geqie-1.0-py3-none-any.whl size=69866 sha256=80868e3d3219cd6d6afeae8f56476868473e3c31c94fd931f412444e55e3442b
  Stored in directory: /tmp/pip-ephem-wheel-cache-190cyhrx/wheels/e8/a4/3c/0d091175f5d58ce0b0ef90bb60820b9014467e3312168c270a
Successfully built geqie
  Attemptin

In [2]:
import sys
QNN_EXPERIMENT_PATH = GEQIE_PATH / "experiments" / "QNN_integration"
sys.path.append(str(QNN_EXPERIMENT_PATH))

from QNN_files import (load_and_process_mnist_dataset,
					   prepare_train_val_precomputed,
					   prepare_test_precomputed,
					   train_QNN,
					   test_QNN
					   )

In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

LABELS_TO_INCLUDE=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
N_SAMPLES_PER_CLASS_TRAIN=100

label_encoder = LabelEncoder()
label_encoder.fit(LABELS_TO_INCLUDE)


X_train_val, y_train_val = load_and_process_mnist_dataset(
	labels_to_include=LABELS_TO_INCLUDE,
	n_samples_per_label=N_SAMPLES_PER_CLASS_TRAIN, 
	resize=(8, 8), 
	path_to_mnist_dataset=path_to_train_mnist_dataset)



Precomputing train:   0%|          | 0/160 [00:00<?, ?it/s]

[train] resume enabled: found 160 existing batches in /content/drive/MyDrive/Colab_Notebooks/GEQIE/circuits/train
Google Colab has 1 workers available.


Computing train batches: 0it [00:00, ?it/s]
Precomputing val:   0%|          | 0/40 [00:00<?, ?it/s]

[val] resume enabled: found 40 existing batches in /content/drive/MyDrive/Colab_Notebooks/GEQIE/circuits/val
Google Colab has 1 workers available.


Computing val batches: 0it [00:00, ?it/s]
Precomputing test:   0%|          | 0/40 [00:00<?, ?it/s]

[test] resume enabled: found 40 existing batches in /content/drive/MyDrive/Colab_Notebooks/GEQIE/circuits/test
Google Colab has 1 workers available.


Computing test batches: 0it [00:00, ?it/s]
Precomputing test: 100%|██████████| 40/40 [00:00<00:00, 4336.54it/s]


In [ ]:
y_train_val = label_encoder.transform(y_train_val)

NUM_CLASSES = np.unique(y_train_val).shape[0]

prepared_train_val = prepare_train_val_precomputed(
	X_train_val,
	y_train_val,
	val_size=0.2,
	batch_size=5,
	save_dir=precomputed_circuits_dir,
	random_state=42,
	stratify=True
)
train_dir = prepared_train_val["train_dir"]
val_dir = prepared_train_val["val_dir"]

X_test, y_test = load_and_process_mnist_dataset(
	labels_to_include=LABELS_TO_INCLUDE, 
	n_samples_per_label=int(N_SAMPLES_PER_CLASS_TRAIN*0.2), 
	resize=(8, 8), 
	path_to_mnist_dataset=path_to_test_mnist_dataset)
y_test = label_encoder.transform(y_test)

prepared_test  = prepare_test_precomputed(
	X_test,
	y_test,
	batch_size=5,
	save_dir=precomputed_circuits_dir,
	random_state=42,
	stratify=True
)
test_dir = prepared_test["test_dir"]

In [ ]:
results_dict = train_QNN(
	epochs=100,
	num_classes=np.unique(y_train_val).shape[0],
	train_dir=train_dir,
	val_dir=val_dir,
	checkpoint_path=checkpoint_path,
	resume_training=True,
)


Training:  12%|█▏        | 12/100 [8:15:11<61:11:27, 2503.27s/it, loss=0.04, val_loss=3.23]

: 

### Test

In [ ]:
test_results = test_QNN(qnn_model=results_dict['qnn_model'], num_classes=np.unique(y_train_val).shape[0], test_dir=test_dir)

### Plots:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(3, 2, figsize=(16, 10))
fig.suptitle(f"Quantum gradient evolution during training for {LABELS_TO_INCLUDE} classification", fontsize=16)
quantum_weights_history = np.array(results_dict['quantum_layer_weights_history'])
for i in range(quantum_weights_history.shape[1]):
	ax[0, 0].plot(quantum_weights_history[:, i], label=f"Theta {i}")
ax[0, 0].set_xlabel("Epoch")
ax[0, 0].set_ylabel("Quantum Weight Value")

classical_weights_history = np.array(results_dict['classical_head_weights_history'])
for i in range(classical_weights_history.shape[1]):
	ax[0, 1].plot(classical_weights_history[:, i], label=f"Weight {i}")
ax[0, 1].set_xlabel("Epoch")
ax[0, 1].set_ylabel("Classical Weight Value")


ax[1, 0].plot(results_dict['train_losses'], label="Train Loss")
ax[1, 0].set_xlabel("Epoch")
ax[1, 0].set_ylabel("Training Loss Value")


ax[1, 1].plot(results_dict['val_losses'], label="Validation Loss")
ax[1, 1].set_xlabel("Epoch")
ax[1, 1].set_ylabel("Validation Loss Value")

cm = test_results["confusion_matrix"]
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS_TO_INCLUDE)
disp.plot(ax=ax[2, 0], cmap=plt.cm.Blues, colorbar=False)

ax[2, 1].axis("off")  # Hide the empty subplot

plt.show()